In [1]:
%run ../stage1.py \
    --sc_file "/mnt/d/ST_Graduation_Project_data/DATA24/scRNA.h5ad" \
    --st_file "/mnt/d/ST_Graduation_Project_data/DATA24/Spatial.h5ad" \
    --n_epochs 300 \
    --resolution 2 \
    --loss_type zinb \
    --beta 0.1 \
    --lambda_mmd 1.0 \
    --top_n_per_type 50 \
    --latent_dim 256 \
    --output_dir ../deconv_results/DATA24 \
    --marker_selection_method l1
 #   --precomputed_marker_file "../deconv_results/CID44971/final_genes.txt"

/home/mwc/miniconda3/envs/dl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cpu
Stage 1 Training: VAE (SC + ST, Marker Genes)
Configuration:
   Marker genes per type: 50
   Leiden resolution: 2.0
   Batch size: 256
   Epochs: 300
   Learning rate: 0.0005
   Beta (KL weight): 0.1
   Hidden dims: [512, 256]
   Latent dim: 256
   Loss type: ZINB
   Lambda MMD: 1.0
   Dual Decoder: True
   Aggregation method: mean
Loading datasets...
   Loading SC: /mnt/d/ST_Graduation_Project_data/DATA24/scRNA.h5ad
   SC shape: (10000, 31489)
   SC avg counts/cell: 933.9 (from X)
   Loading ST: /mnt/d/ST_Graduation_Project_data/DATA24/Spatial.h5ad
   ST shape: (1000, 27345)
   Common genes: 17733
   SC final: (10000, 17733)
   ST final: (1000, 17733)
Computing clusters and marker genes...
Starting clustering analysis...
Clustering results: 32 clusters
Marker genes per cluster:
   0: 50 -> 50 (after L1)
   1: 50 -> 50 (after L1)
   10: 50 -> 50 (after L1)
   11: 50 -> 50 (after L1)
   12: 50 -> 50 (after L1)
   13: 50 -> 50 (after L1)
   14: 50 -> 50 (after L1)
   15

VAE Training:  45%|████▌     | 136/300 [02:36<03:08,  1.15s/epoch, Train=698.5761, Recon=694.8680, KL=37.0799, MMD=0.0317, Test=767.0593]



⚠️ Early stopping triggered at epoch 137/300
   Best test loss: 766.2490, Current test loss: 767.0593
   No improvement for 20 consecutive epochs
📊 Computing cluster centers using ALL SC data (train + test)...
   ALL SC data: (10000, 681)
   Number of clusters: 32
   Computed embeddings shape: (10000, 256)
Computing cluster centers with mean aggregation...
      Cluster 0: 976 cells (mean aggregation)
      Cluster 1: 834 cells (mean aggregation)
      Cluster 2: 374 cells (mean aggregation)
      Cluster 3: 359 cells (mean aggregation)
      Cluster 4: 350 cells (mean aggregation)
      Cluster 5: 325 cells (mean aggregation)
      Cluster 6: 293 cells (mean aggregation)
      Cluster 7: 287 cells (mean aggregation)
      Cluster 8: 287 cells (mean aggregation)
      Cluster 9: 283 cells (mean aggregation)
      Cluster 10: 277 cells (mean aggregation)
      Cluster 11: 237 cells (mean aggregation)
      Cluster 12: 599 cells (mean aggregation)
      Cluster 13: 183 cells (mean aggre

In [2]:
%run ../stage2.py \
    --st_file "/mnt/d/ST_Graduation_Project_data/DATA24/Spatial.h5ad" \
    --stage1_model_path "../deconv_results/DATA24/final_vae.pth" \
    --output_dir "../deconv_results/DATA24/" \
    --gat_hidden_dim 512 \
    --gat_layers 4 \
    --lr 1e-4 \
    --k_spatial 10 \
    --k_celltype 20 \
    --batch_size 512 \
    --n_epochs 500 \
    --weight_threshold 0.05

Sample name: Spatial
Stage 1 model: ../deconv_results/DATA24/final_vae.pth
Output directory: ../deconv_results/DATA24/
Device: cpu
Weight threshold: 0.05
Loading pretrained VAE Encoder...
   VAE architecture: 681 -> 256
   Output type: zinb
   Architecture: Dual Decoder (SC/ST-specific)
   ✓ Loaded 10000 cell cluster labels from checkpoint
Loading cluster data from: ../deconv_results/DATA24/final_vae_cluster_data.npz
   Cluster prototypes: torch.Size([32, 256])
   Cluster expressions (marker): torch.Size([32, 681])
   Cluster expressions (all genes): 32 × 17733
   Loaded celltype mapping: 32 clusters
   Average cell counts: 933.9
Loaded all genes list: 17733 genes
VAE Encoder loaded: 681 -> 256
Cell type clusters: ['0', '1', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '2', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '3', '30', '31', '4', '5', '6', '7', '8', '9']
Marker genes: 681
Using Stage 1 cluster centers and expressions...
Loaded 32 clusters
Using S

GAT Training: 100%|██████████| 500/500 [14:13<00:00,  1.71s/epoch, Total=4.7534, MSE=27.9236, Spot_Cosine=0.1896, Gene_Cosine=0.3672, Pearson=0.3047, Gene_Pearson=0.6937, P_pat=14, M_pat=14, C_pat=14]


Evaluating model results...
Applying weight threshold: 0.05
   Non-zero elements: 20000 -> 4874 (15.2%)
Saving deconvolution results...
Generating deconvolution expression matrices...
   Reconstructing all gene expression...
   Saved: ../deconv_results/DATA24//Spatial_reconstructed_all_genes.csv
   Cell type composition...
   Found duplicate celltype names: ['Thick Ascending Limb', 'Intercalated Cells Type A', 'Proximal Tubule', 'Endothelial Cell', 'Principal Cells']. Merging corresponding cluster columns by summing weights.
   Columns before: 32, after merge: 10
   Saved cell composition (celltype): ../deconv_results/DATA24//Spatial_cell_composition.csv
   Saved cluster composition: ../deconv_results/DATA24//Spatial_cluster_composition.csv
   Using 681 marker genes for reconstruction quality (consistent with training objective)
   Cosine similarities saved: ../deconv_results/DATA24//Spatial_spot_cosine_similarity.csv

Plotting reconstruction quality curve...
   Reconstruction quality 

In [4]:
import scanpy as sc

adata = sc.read_h5ad("/mnt/d/ST_Graduation_Project_data/DATA24/Spatial.h5ad")
adata

AnnData object with n_obs × n_vars = 1000 × 27345
    obs: 'cell_counts'
    uns: 'density'

In [5]:
print(adata.uns['density'])
df_density = adata.uns['density']
df_density.to_csv("../evaluate/cell_type_density.csv")

     Principal Cells  Proximal Tubule  Thick Ascending Limb  \
0                0.0              2.0                   6.0   
1                0.0              4.0                   4.0   
2                0.0              3.0                   3.0   
3                2.0              0.0                   4.0   
4                0.0              8.0                   0.0   
..               ...              ...                   ...   
995              0.0              6.0                   8.0   
996              0.0              2.0                   3.0   
997              0.0              2.0                   3.0   
998              0.0              1.0                   5.0   
999              0.0              5.0                   0.0   

     Distal Convoluted Tubule  Intercalated Cells Type A  \
0                         2.0                        2.0   
1                         0.0                        2.0   
2                         0.0                        0.0   
3  